Tailor dim table containing exam periods for UHelsinki from 2023 to 2026

In [ ]:
%cd ../../

In [ ]:
import polars as pl

# Craft the dim table

In [ ]:
DATE_START = pl.lit('2023-01-01').str.to_date()
DATE_END = pl.lit('2026-12-31').str.to_date()

dim_exam_raw = (
    pl
    .from_records([
        {'date_begin': '2023-03-06', 'date_end': '2023-03-12'},
        {'date_begin': '2023-05-01', 'date_end': '2023-05-07'},
        {'date_begin': '2023-10-23', 'date_end': '2023-10-29'},
        {'date_begin': '2023-12-18', 'date_end': '2023-12-24'},

        {'date_begin': '2024-03-04', 'date_end': '2024-03-10'},
        {'date_begin': '2024-05-06', 'date_end': '2024-05-12'},
        {'date_begin': '2024-10-21', 'date_end': '2024-10-27'},
        {'date_begin': '2024-12-16', 'date_end': '2024-12-22'},

        {'date_begin': '2025-03-03', 'date_end': '2025-03-09'},
        {'date_begin': '2025-05-05', 'date_end': '2025-05-11'},
        {'date_begin': '2025-10-20', 'date_end': '2025-10-26'},
        {'date_begin': '2025-12-15', 'date_end': '2025-12-21'},

        {'date_begin': '2026-03-02', 'date_end': '2026-03-08'},
        {'date_begin': '2026-05-04', 'date_end': '2026-05-10'},
        {'date_begin': '2026-10-19', 'date_end': '2026-10-25'},
        {'date_begin': '2026-12-14', 'date_end': '2026-12-20'},
    ])
    .with_columns(
        pl.col('date_begin').str.to_date(),
        pl.col('date_end').str.to_date(),
    )
)  # fmt: skip


df = (
    pl.DataFrame()

    # Create blank dataframe with date
    .with_columns(
        pl.date_range(DATE_START, DATE_END, '1d').alias('date')
    )
)

entries_exam = (
    df
    .join(dim_exam_raw, how='cross')
    .filter(
        (pl.col('date') >= pl.col('date_begin'))
        & (pl.col('date') <= pl.col('date_end'))
    )
    .select(
        'date',
        pl.lit(True).alias('is_exam')
    )
)
dim_exams_uhelsinki = (
    df
    .join(entries_exam, on='date', how='left')
    .with_columns(pl.col('is_exam').fill_null(False))
)

dim_exams_uhelsinki.head()

## Check

In [ ]:
days_exam_202303 = (
    dim_exams_uhelsinki
    .filter(
        (1 == 1)
        & (pl.col('date').dt.year() == 2023)
        & (pl.col('date').dt.month() == 3)
        & (pl.col('is_exam'))
    )
    # .select(pl.col('date').dt.day())
    ['date'].dt.day().to_list()
)

assert days_exam_202303 == [6, 7, 8, 9, 10, 11, 12]

# Save

In [ ]:
path = "data/processed/dim_exams_uhelsinki.xlsx"
dim_exams_uhelsinki.write_excel(path)